In [8]:
import numpy as np


def compute_NOA_massa(WL, Spectrum):
    """
    Compute the NOA (Normalized Olivine Abundance) from:
    - WL: array of wavelengths
    - Spectrum: array of reflectance values
    
    Returns:
        NOA

    Author: G. Massa
    Reference: Massa, Giuseppe. "Spectral studies in view of the Hera mission." (2025). PhD Thesis. https://hdl.handle.net/20.500.14242/189873
    """

    # Convert to float32
    WL = WL.astype(np.float32)
    Spectrum = Spectrum.astype(np.float32)

    # Step 1: find indices
    value1 = np.where(WL > 0.65)[0][0]
    value2 = np.where(WL > 1.8)[0][0]
    value3 = np.where(WL > 0.95)[0][0]
    value4 = np.where(WL > 0.8)[0][0] - value1
    value5 = np.where(WL > 1.2)[0][0] - value1
    value6 = value3 - value1

    # Step 2: maxima of the shoulders
    max_1 = value1 + np.max(np.where(Spectrum[value1:value3+1] == np.max(Spectrum[value1:value3+1]))[0])
    max_2 = value3 + np.max(np.where(Spectrum[value3:value2+1] == np.max(Spectrum[value3:value2+1]))[0])

    # Step 3: linear fit
    slope = (Spectrum[max_2] - Spectrum[max_1]) / (WL[max_2] - WL[max_1])
    intercept = Spectrum[max_1] - slope * WL[max_1]

    # Step 4: continuum
    continuum = intercept + slope * WL[value1:value2+1]

    # Step 5: continuum removal
    corrected = Spectrum[value1:value2+1] / continuum

    # Step 6: band depth calculation
    subrange = corrected[value4:value5+1]
    posit = np.max(np.where(subrange == np.min(subrange))[0])
    banddepth1 = 1 - corrected[posit + value4]

    # Step 7: band area calculation
    BandArea = 0.0
    for l in range(1, len(WL[max_1:max_2+1])):
        delta = WL[max_1 + l] - WL[max_1 + l - 1]
        BandArea += delta - delta * corrected[max_1 + l - value1]

    # Step 8: DA1 and NOA calculation
    DA1 = (banddepth1 - 0.008) / (BandArea + 0.002)
    NOA = 0.916 - 0.120 * DA1

    return NOA
